# JASP → MLIR demo

This notebook demonstrates the **JASP MLIR emission pipeline** from the `jasp_next` branch of Qrisp.

It shows how a Qrisp quantum program is:
1. Traced to a JAX expression (`jaspr`)
2. Lowered to MLIR with proper JASP dialect ops
3. Structured so that quantum kernels are plain `func.func` ops (identified by `!jasp.QuantumState` in their signature), called via `jasp.call` which presents a classical-only interface

No local install needed — everything runs here in Colab.

## 0. Install

In [ ]:
%pip uninstall torch sympy -y
%pip install sympy==1.13
%pip install xdsl
%pip install -q git+https://github.com/bruno-ah-um/Qrisp.git@jasp_next

## 1. Basic MLIR emission

A simple circuit: allocate 3 qubits, apply H + CX, measure. The resulting MLIR shows `jasp.create_qubits`, `jasp.quantum_gate`, and `jasp.measure` printed without quotes — confirming the JASP dialect is properly registered in xDSL.

The pipeline automatically extracts `@main`'s quantum body into a `func.func private @main_kernel` (a quantum kernel identified by `!jasp.QuantumState` as its last input/output) and inserts a `jasp.call` wrapper in `@main`.

In [1]:
from qrisp import QuantumVariable, h, cx, measure
from qrisp.jasp import make_jaspr

def main():
    qv = QuantumVariable(3)
    h(qv[0])
    cx(qv[0], qv[1])
    cx(qv[0], qv[2])
    return measure(qv)

jaspr = make_jaspr(main)()
xdsl_module = jaspr.to_mlir()
print(xdsl_module)

builtin.module @jasp_module {
  func.func @main() -> tensor<i64> {
    %0 = jasp.call @main_kernel() : () -> tensor<i64>
    func.return %0 : tensor<i64>
  }
  func.func private @main_kernel(%arg14 : !jasp.QuantumState) -> (tensor<i64>, !jasp.QuantumState) {
    %0 = "stablehlo.constant"() <{value = dense<3> : tensor<i64>}> : () -> tensor<i64>
    %1, %2 = jasp.create_qubits %0, %arg14 : !jasp.QuantumState, tensor<i64> -> !jasp.QubitArray, !jasp.QuantumState
    %3 = "stablehlo.constant"() <{value = dense<0> : tensor<i64>}> : () -> tensor<i64>
    %4 = jasp.get_qubit %1, %3 : !jasp.QubitArray, tensor<i64> -> !jasp.Qubit
    %5 = jasp.quantum_gate "h" (%4) , %2 : (!jasp.Qubit) , !jasp.QuantumState -> !jasp.QuantumState
    %6 = "stablehlo.constant"() <{value = dense<1> : tensor<i64>}> : () -> tensor<i64>
    %7 = jasp.get_qubit %1, %6 : !jasp.QubitArray, tensor<i64> -> !jasp.Qubit
    %8 = jasp.quantum_gate "cx" (%4, %7) , %5 : (!jasp.Qubit, !jasp.Qubit) , !jasp.QuantumState -> !jasp.Qu

## 2. Quantum control flow

JAX normally emits `stablehlo.case` / `stablehlo.while` for control flow. The `fix_quantum_control_flow` pass rewrites these to `scf.if` / `scf.while` when they carry `!jasp.QuantumState` — necessary because `stablehlo` control flow cannot hold quantum types.

In [2]:
from qrisp import QuantumVariable, QuantumFloat, h, x, rz, cx, measure, control
from qrisp.jasp import make_jaspr, jrange

def main():
    qv = QuantumVariable(3)
    h(qv[0])
    c = measure(qv[0])          # classical bit from mid-circuit measurement
    with control(c):             # measurement-controlled gate
        x(qv[1])
    for i in jrange(qv.size):   # dynamic loop over qubit count
        h(qv[i])

jaspr = make_jaspr(main)()
xdsl_module = jaspr.to_mlir()
mlir_str = str(xdsl_module)

print("stablehlo.case  present:", "stablehlo.case"  in mlir_str)  # should be False
print("stablehlo.while present:", "stablehlo.while" in mlir_str)  # should be False
print("scf.if present:         ", "scf.if"          in mlir_str)  # should be True
print()
print(mlir_str)

stablehlo.case  present: False
stablehlo.while present: False
scf.if present:          True

builtin.module @jasp_module {
  func.func @main() {
    %0, %1, %2, %3 = jasp.call @main_kernel() : () -> (tensor<i1>, tensor<i64>, tensor<i64>, !jasp.QubitArray)
    %4 = "stablehlo.convert"(%0) : (tensor<i1>) -> tensor<i32>
    %5 = tensor.extract %4[] : tensor<i32>
    %6 = arith.constant 0 : i32
    %7 = arith.cmpi ne, %5, %6 : i32
    %8 = scf.if %7 -> (!jasp.QuantumState) {
      scf.yield %9 : !jasp.QuantumState
    } else {
      %10 = "stablehlo.constant"() <{value = dense<1> : tensor<i64>}> : () -> tensor<i64>
      %11 = jasp.get_qubit %12, %10 : !jasp.QubitArray, tensor<i64> -> !jasp.Qubit
      %13 = jasp.quantum_gate "x" (%11) , %9 : (!jasp.Qubit) , !jasp.QuantumState -> !jasp.QuantumState
      scf.yield %13 : !jasp.QuantumState
    }
    %14 = "stablehlo.subtract"(%1, %2) : (tensor<i64>, tensor<i64>) -> tensor<i64>
    %15 = "stablehlo.subtract"(%14, %14) : (tensor<i64>, tensor<

## 3. `quantum_kernel`: QPU / CPU separation

The `@quantum_kernel` decorator marks a function as QPU code. The pipeline:

1. **`lift_quantum_kernels`** — replaces the `create_quantum_kernel` / `func.call` / `consume_quantum_kernel` sentinel triplet with a classical `jasp.call`. The callee remains a regular `func.func` — it is identified as a quantum kernel by having `!jasp.QuantumState` as its last input and output.
2. **`hoist_classical_ops`** — moves non-QPU-safe ops (e.g. `stablehlo.convert`, `stablehlo.multiply`) out of the kernel body and into `@main`, where they run on the CPU host after the QPU returns.
3. **`drop_dead_wrappers`** — erases unused `private func.func` stubs emitted by JAX.

The result: the kernel body contains **only** `jasp.*` ops and `stablehlo.constant`; everything else is in `@main`. Only `jasp.call` is a new op — quantum kernels reuse `func.func` and `func.return` from the existing MLIR `func` dialect.

In [3]:
from qrisp import QuantumFloat, measure
from qrisp.jasp import make_jaspr, quantum_kernel
from qrisp.jasp.mlir.mlir_emission import jaspr_to_mlir
import jax.numpy as jnp

@quantum_kernel
def sampling_kernel(k):
    """Runs on the QPU: allocate qubits, apply gates, measure."""
    qf = QuantumFloat(4)
    return measure(qf)

def main(k):
    return sampling_kernel(k)

jaspr = make_jaspr(main)(jnp.int64(1))
xdsl_module = jaspr_to_mlir(jaspr)
mlir_str = str(xdsl_module)

print("func.func kernel present:", "func.func private @sampling_kernel" in mlir_str)
print("jasp.call present:       ", "jasp.call"    in mlir_str)
print("func.return present:     ", "func.return"  in mlir_str)
print()
print(mlir_str)

func.func kernel present: True
jasp.call present:        True
func.return present:      True

builtin.module @jasp_module {
  func.func public @main(%arg11 : tensor<i64>, %arg12 : !jasp.QuantumState) -> (tensor<f64>, !jasp.QuantumState) {
    %0, %1 = jasp.call @sampling_kernel(%arg11) : (tensor<i64>) -> (tensor<i64>, tensor<f64>)
    %2 = "stablehlo.convert"(%0) : (tensor<i64>) -> tensor<f64>
    %3 = "stablehlo.multiply"(%2, %1) : (tensor<f64>, tensor<f64>) -> tensor<f64>
    func.return %3, %arg12 : tensor<f64>, !jasp.QuantumState
  }
  func.func private @sampling_kernel(%arg9 : tensor<i64>, %arg10 : !jasp.QuantumState) -> (tensor<i64>, tensor<f64>, !jasp.QuantumState) {
    %0 = "stablehlo.constant"() <{value = dense<4> : tensor<i64>}> : () -> tensor<i64>
    %1, %2 = jasp.create_qubits %0, %arg10 : !jasp.QuantumState, tensor<i64> -> !jasp.QubitArray, !jasp.QuantumState
    %3, %4 = jasp.measure %1, %2 : !jasp.QubitArray, !jasp.QuantumState -> tensor<i64>, !jasp.QuantumState
    %5

## 4. EmitC lowering: preparing for C emission

The `jaspr_to_emitc` pipeline extends the standard MLIR pipeline with three additional passes:

1. **`strip_quantum_state_from_main`** — removes the `!jasp.QuantumState` argument/result from `@main` (an artefact of the JAX tracing convention), making it purely classical.
2. **`lower_jasp_call_to_qdmi`** — serialises each quantum kernel `func.func` as an MLIR string and replaces `jasp.call` with `emitc.call_opaque "run_jasp_kernel"(kernel_mlir, args...)`.
3. **`lower_classical_to_emitc`** — rewrites `stablehlo.{convert,multiply,add,subtract,constant}` to `emitc.{cast,mul,add,sub,constant}`.

The result is a module that consists entirely of EmitC dialect ops — ready for translation to C++.

In [4]:
from qrisp import QuantumFloat, measure
from qrisp.jasp import make_jaspr, quantum_kernel
from qrisp.jasp.mlir.cpp_emission import jaspr_to_emitc
import jax.numpy as jnp

@quantum_kernel
def sampling_kernel(k):
    """Runs on the QPU: allocate qubits, apply gates, measure."""
    qf = QuantumFloat(4)
    return measure(qf)

def main(k):
    return sampling_kernel(k)

jaspr = make_jaspr(main)(jnp.int64(1))
emitc_module = jaspr_to_emitc(jaspr)
mlir_str = str(emitc_module)

print("emitc.call_opaque present:", "emitc.call_opaque" in mlir_str)
print("emitc.cast present:       ", "emitc.cast"        in mlir_str)
print("emitc.mul present:        ", "emitc.mul"         in mlir_str)
print("jasp.call absent:         ", "jasp.call" not in mlir_str)
print()
print(mlir_str)

emitc.call_opaque present: True
emitc.cast present:        True
emitc.mul present:         True
jasp.call absent:          True

builtin.module @jasp_module {
  func.func public @main(%arg11 : tensor<i64>) -> (tensor<f64>) {
    %0, %1 = emitc.call_opaque "run_jasp_kernel"(%arg11) {args = [#emitc.opaque<"\"func.func private @sampling_kernel(%arg9 : tensor<i64>, %arg10 : !jasp.QuantumState) -> (tensor<i64>, tensor<f64>, !jasp.QuantumState) {\\n  %0 = \\\"stablehlo.constant\\\"() <{value = dense<4> : tensor<i64>}> : () -> tensor<i64>\\n  %1, %2 = jasp.create_qubits %0, %arg10 : !jasp.QuantumState, tensor<i64> -> !jasp.QubitArray, !jasp.QuantumState\\n  %3, %4 = jasp.measure %1, %2 : !jasp.QubitArray, !jasp.QuantumState -> tensor<i64>, !jasp.QuantumState\\n  %5 = \\\"stablehlo.constant\\\"() <{value = dense<2.000000e+00> : tensor<f64>}> : () -> tensor<f64>\\n  %6 = \\\"stablehlo.constant\\\"() <{value = dense<1.000000e+00> : tensor<f64>}> : () -> tensor<f64>\\n  func.return %3, %6, %4 : t

## 5. C emission via QDMI

`jaspr.to_cpp()` compiles the full pipeline and translates the EmitC IR to C++ using `mlir-translate --mlir-to-cpp`.

The generated C file is **self-contained** — it inlines the full QDMI runtime so you get a single `.c` file that is ready to compile. The file includes:

- A **compile command** as the first comment line
- The inlined runtime source (`runtime.h` + `runtime_internal.h` + `runtime.c`)
- `runtime_init()` at the start of `main()`
- `runtime_cleanup()` before every `return` in `main()`
- `run_jasp_kernel()` calls that submit the quantum kernel's MLIR (embedded as a C raw string literal) to the QPU via QDMI

The only external dependency at compile time is the QDMI SDK (headers + driver library).

In [5]:
from qrisp import QuantumFloat, measure
from qrisp.jasp import make_jaspr, quantum_kernel
import jax.numpy as jnp

@quantum_kernel
def sampling_kernel(k):
    """Runs on the QPU: allocate qubits, apply gates, measure."""
    qf = QuantumFloat(4)
    return measure(qf)

def main(k):
    return sampling_kernel(k)

jaspr = make_jaspr(main)(jnp.int64(1))
cpp_code = jaspr.to_cpp()
print(cpp_code)

// Compile with (set QDMI_ROOT to your QDMI checkout):
//   clang -std=c11 -I$QDMI_ROOT/include -I$QDMI_ROOT/examples/driver \
//     thisfile.c $QDMI_ROOT/build/examples/driver/libqdmi_example_driver.a \
//     -ldl -lstdc++ -o program

#include <stdint.h>

/*
********************************************************************************
* Copyright (c) 2026 the Qrisp authors
*
* This program and the accompanying materials are made available under the
* terms of the Eclipse Public License 2.0 which is available at
* http://www.eclipse.org/legal/epl-2.0.
*
* This Source Code may also be made available under the following Secondary
* Licenses when the conditions for such availability set forth in the Eclipse
* Public License, v. 2.0 are satisfied: GNU General Public License, version 2
* with the GNU Classpath Exception which is
* available at https://www.gnu.org/software/classpath/license.html.
*
* SPDX-License-Identifier: EPL-2.0 OR GPL-2.0 WITH Classpath-exception-2.0
**************

### Implicit kernel (no decorator)

When no explicit `@quantum_kernel` is used, the pipeline extracts `@main`'s quantum body into `@main_kernel` automatically. The generated C++ is the same shape — a single `run_jasp_kernel()` call.

In [6]:
from qrisp import QuantumVariable, h, cx, measure
from qrisp.jasp import make_jaspr

def bell_state():
    qv = QuantumVariable(2)
    h(qv[0])
    cx(qv[0], qv[1])
    return measure(qv)

jaspr = make_jaspr(bell_state)()
print(jaspr.to_cpp())

// Compile with (set QDMI_ROOT to your QDMI checkout):
//   clang -std=c11 -I$QDMI_ROOT/include -I$QDMI_ROOT/examples/driver \
//     thisfile.c $QDMI_ROOT/build/examples/driver/libqdmi_example_driver.a \
//     -ldl -lstdc++ -o program

#include <stdint.h>

/*
********************************************************************************
* Copyright (c) 2026 the Qrisp authors
*
* This program and the accompanying materials are made available under the
* terms of the Eclipse Public License 2.0 which is available at
* http://www.eclipse.org/legal/epl-2.0.
*
* This Source Code may also be made available under the following Secondary
* Licenses when the conditions for such availability set forth in the Eclipse
* Public License, v. 2.0 are satisfied: GNU General Public License, version 2
* with the GNU Classpath Exception which is
* available at https://www.gnu.org/software/classpath/license.html.
*
* SPDX-License-Identifier: EPL-2.0 OR GPL-2.0 WITH Classpath-exception-2.0
**************

## 6. Build & run via QDMI

Since `to_cpp()` produces a self-contained C file (runtime included), building is a single `clang` invocation. The compile command is embedded as the first comment in the generated file.

Prerequisites (local machine only):
- `clang` with C11 support
- `mlir-translate` (MLIR/LLVM tools)
- QDMI built at `$QDMI_ROOT` with example driver & device libs

In [7]:
import os

QDMI_ROOT = os.environ.get("QDMI_ROOT", os.path.expanduser("~/git/QDMI"))
OUTPUT_PATH = os.path.expanduser("~/jasp_bell.c")

# Generate the self-contained C file from the bell-state example
from qrisp import QuantumVariable, h, cx, measure
from qrisp.jasp import make_jaspr

def bell_state():
    qv = QuantumVariable(2)
    h(qv[0])
    cx(qv[0], qv[1])
    return measure(qv)

jaspr = make_jaspr(bell_state)()
cpp_code = jaspr.to_cpp()

with open(OUTPUT_PATH, "w") as f:
    f.write(cpp_code)

print(f"Written to {OUTPUT_PATH}\n")
print(cpp_code)

Written to /home/bca/jasp_bell.c

// Compile with (set QDMI_ROOT to your QDMI checkout):
//   clang -std=c11 -I$QDMI_ROOT/include -I$QDMI_ROOT/examples/driver \
//     thisfile.c $QDMI_ROOT/build/examples/driver/libqdmi_example_driver.a \
//     -ldl -lstdc++ -o program

#include <stdint.h>

/*
********************************************************************************
* Copyright (c) 2026 the Qrisp authors
*
* This program and the accompanying materials are made available under the
* terms of the Eclipse Public License 2.0 which is available at
* http://www.eclipse.org/legal/epl-2.0.
*
* This Source Code may also be made available under the following Secondary
* Licenses when the conditions for such availability set forth in the Eclipse
* Public License, v. 2.0 are satisfied: GNU General Public License, version 2
* with the GNU Classpath Exception which is
* available at https://www.gnu.org/software/classpath/license.html.
*
* SPDX-License-Identifier: EPL-2.0 OR GPL-2.0 WITH Clas

In [8]:
import subprocess, shutil

# Build the compile command (same as the comment in the generated file)
clang_cmd = (
    f"clang -std=c11 "
    f"-I{QDMI_ROOT}/include -I{QDMI_ROOT}/examples/driver "
    f"{OUTPUT_PATH} "
    f"{QDMI_ROOT}/build/examples/driver/libqdmi_example_driver.a "
    f"-ldl -lstdc++ -o jasp_bell"
)

print(f"$ {clang_cmd}\n")
result = subprocess.run(clang_cmd, shell=True, capture_output=True, text=True)
if result.stdout:
    print(result.stdout)
if result.returncode != 0:
    print("Compile errors:\n", result.stderr)
    raise SystemExit(result.returncode)
print("Compiled successfully.\n")

# Copy qdmi.conf next to the binary so the driver finds the device
shutil.copy(os.path.join(QDMI_ROOT, "demo", "qdmi.conf"), ".")

# Run
print("$ ./jasp_bell\n")
result = subprocess.run(["./jasp_bell"], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("stderr:", result.stderr)
if result.returncode != 0:
    print(f"Exit code: {result.returncode}")

$ clang -std=c11 -I/home/bca/git/QDMI/include -I/home/bca/git/QDMI/examples/driver /home/bca/jasp_bell.c /home/bca/git/QDMI/build/examples/driver/libqdmi_example_driver.a -ldl -lstdc++ -o jasp_bell

Compiled successfully.

$ ./jasp_bell


stderr: Couldn't open the device: Couldn't open the device library: ../build/examples/device/libcxx_device.so
runtime: QDMI driver init failed
runtime: not initialized

Exit code: 255
